# FAISS Similarity Search Tutorial

This notebook is a Colab-friendly version of `faiss.py`. Run the cells from top to bottom to:

- install the required libraries
- generate sample vectors
- compare brute-force cosine search with FAISS `IndexFlatL2`
- run FAISS `IndexIVFFlat` with regions
- visualize the results after each step
- run a lighter benchmark cell at the end


## 1. Install dependencies

Run this once in Google Colab before the rest of the notebook.

In [ ]:
!pip install faiss-cpu matplotlib scikit-learn


## 2. Imports and helper functions

Cosine similarity compares how closely two vectors point in the same direction. FAISS then gives us faster search structures, including exact search with `IndexFlatL2` and approximate region-based search with `IndexIVFFlat`.

In [ ]:
import time

import faiss
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import normalize


def generate_sample_vectors(n_vectors=100, n_dimensions=3, seed=42):
    """Generate synthetic vectors for demonstration."""
    np.random.seed(seed)
    vectors = np.random.randn(n_vectors, n_dimensions)
    normalized_vectors = normalize(vectors, norm='l2')
    return normalized_vectors


def plot_vectors_3d(vectors, query_vector=None, matches=None, title="Vector Space Visualization"):
    """Basic 3D visualization without regions."""
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(vectors[:, 0], vectors[:, 1], vectors[:, 2], c='blue', alpha=0.5, label='Database vectors')

    if query_vector is not None:
        ax.scatter(query_vector[0], query_vector[1], query_vector[2], c='red', s=100, label='Query vector')

    if matches is not None:
        match_vectors = vectors[matches]
        ax.scatter(match_vectors[:, 0], match_vectors[:, 1], match_vectors[:, 2], c='green', s=100, label='Matches')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    plt.show()


def compute_vector_assignments(vectors, centroids):
    """Compute which vectors belong to which centroids."""
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(centroids)
    distances, assignments = index.search(vectors, 1)
    return distances, assignments.ravel()


def plot_vectors_with_regions(vectors, centroids, query_vector=None, matches=None, searched_regions=None, title="Vector Space with FAISS Regions"):
    """Visualize vectors in 3D space with their clusters or regions."""
    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111, projection='3d')

    distances, assignments = compute_vector_assignments(vectors, centroids)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(centroids)))

    for i in range(len(centroids)):
        cluster_vectors = vectors[assignments == i]
        if len(cluster_vectors) > 0:
            alpha = 1.0 if searched_regions is None or i in searched_regions else 0.1
            ax.scatter(cluster_vectors[:, 0], cluster_vectors[:, 1], cluster_vectors[:, 2], c=[colors[i]], alpha=alpha, label=f'Region {i}')

    ax.scatter(centroids[:, 0], centroids[:, 1], centroids[:, 2], c='black', s=100, marker='*', label='Region Centers')

    if query_vector is not None:
        ax.scatter(query_vector[0], query_vector[1], query_vector[2], c='red', s=200, marker='x', label='Query Vector')

    if matches is not None:
        match_vectors = vectors[matches]
        ax.scatter(match_vectors[:, 0], match_vectors[:, 1], match_vectors[:, 2], c='green', s=100, label='Matches')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    plt.show()


def train_kmeans_get_centroids(vectors, n_clusters):
    """Train k-means and get centroids."""
    kmeans = faiss.Kmeans(d=vectors.shape[1], k=n_clusters, niter=20, verbose=False)
    kmeans.train(vectors)
    return kmeans.centroids


def brute_force_cosine_search(database_vectors, query_vector, k=5):
    """Perform brute force cosine similarity search."""
    start_time = time.time()
    similarities = np.dot(database_vectors, query_vector)
    top_k_indices = np.argsort(similarities)[-k:][::-1]
    end_time = time.time()
    return top_k_indices, end_time - start_time


def faiss_flat_l2_search(database_vectors, query_vector, k=5):
    """Perform basic FAISS L2 search."""
    dimension = database_vectors.shape[1]
    index = faiss.IndexFlatL2(dimension)

    start_time = time.time()
    index.add(database_vectors)
    distances, indices = index.search(query_vector.reshape(1, -1), k)
    end_time = time.time()

    return indices[0], end_time - start_time


def faiss_ivf_search_realistic(database_vectors, query_vectors, k=5, n_regions=10, nprobe=3):
    """Train and search a FAISS IVF index for batch queries."""
    dimension = database_vectors.shape[1]
    print('Training index (this is usually done once)...')

    quantizer = faiss.IndexFlatL2(dimension)
    index = faiss.IndexIVFFlat(quantizer, dimension, n_regions, faiss.METRIC_L2)

    train_start = time.time()
    index.train(database_vectors)
    train_time = time.time() - train_start

    add_start = time.time()
    index.add(database_vectors)
    add_time = time.time() - add_start

    index.nprobe = nprobe

    search_start = time.time()
    distances, indices = index.search(query_vectors, k)
    search_time = time.time() - search_start

    return indices, search_time, train_time, add_time


## 3. Generate sample data and visualize the vector space

In [ ]:
n_vectors = 1000
n_dimensions = 3
k = 5

print(f'Generating {n_vectors} vectors with {n_dimensions} dimensions...')
database_vectors = generate_sample_vectors(n_vectors, n_dimensions)
query_vector = generate_sample_vectors(1, n_dimensions)[0]
query_vector_batch = query_vector.reshape(1, -1)

plot_vectors_3d(database_vectors, query_vector, title='Initial Vector Space')


## 4. Brute-force cosine similarity search

In [ ]:
cosine_matches, cosine_time = brute_force_cosine_search(database_vectors, query_vector, k)
print(f'Brute force search time: {cosine_time:.6f} seconds')
print(f'Top {k} cosine similarity matches (indices): {cosine_matches}')


In [ ]:
plot_vectors_3d(
    database_vectors,
    query_vector,
    cosine_matches,
    'Brute Force Cosine Similarity Results'
)


## 5. Exact FAISS search with `IndexFlatL2`

In [ ]:
faiss_matches, faiss_time = faiss_flat_l2_search(database_vectors, query_vector, k)
print(f'FAISS L2 search time: {faiss_time:.6f} seconds')
print(f'Top {k} L2 distance matches (indices): {faiss_matches}')


In [ ]:
plot_vectors_3d(
    database_vectors,
    query_vector,
    faiss_matches,
    'FAISS L2 Search Results'
)


## 6. Approximate FAISS search with `IndexIVFFlat`

This version clusters vectors into regions first, then searches only the most relevant regions.

In [ ]:
n_regions = 10
nprobe = 3
dimension = database_vectors.shape[1]

quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, n_regions, faiss.METRIC_L2)

print('Training index...')
train_start = time.time()
index.train(database_vectors)
train_time = time.time() - train_start
print(f'Training time: {train_time:.6f} seconds')

print('Adding vectors...')
add_start = time.time()
index.add(database_vectors)
add_time = time.time() - add_start
print(f'Adding time: {add_time:.6f} seconds')

index.nprobe = nprobe

print('Searching...')
search_start = time.time()
distances, ivf_matches = index.search(query_vector_batch, k)
search_time = time.time() - search_start
ivf_matches = ivf_matches[0]

centroids = train_kmeans_get_centroids(database_vectors, n_regions)
_, searched_regions = quantizer.search(query_vector_batch, nprobe)
searched_regions = searched_regions[0]

print(f'Search time: {search_time:.6f} seconds')
print(f'Total time (train + add + search): {train_time + add_time + search_time:.6f} seconds')
print(f'Search-only time: {search_time:.6f} seconds')
print(f'Searched {nprobe} out of {n_regions} regions: {searched_regions}')


In [ ]:
plot_vectors_with_regions(
    database_vectors,
    centroids,
    query_vector,
    ivf_matches,
    searched_regions,
    'FAISS IVF Search Results (Highlighted Searched Regions)'
)


## 7. Compare the match overlap between the methods

In [ ]:
common_matches_basic = set(cosine_matches).intersection(set(faiss_matches))
common_matches_ivf = set(cosine_matches).intersection(set(ivf_matches))

print('Comparing results between methods:')
print(f'Common matches (Cosine vs Basic FAISS): {len(common_matches_basic)}')
print(f'Common match indices: {common_matches_basic}')
print(f'Common matches (Cosine vs IVF FAISS): {len(common_matches_ivf)}')
print(f'Common match indices: {common_matches_ivf}')


## 8. Benchmark across larger datasets

This is the heaviest part of the notebook. The original script uses `n_queries = 1000`; the Colab notebook uses a smaller default so the cell remains interactive. Increase it later if you want to match the original script more closely.

In [ ]:
vector_sizes = [100, 1000, 10000, 100000]
n_queries = 200
k = 5
dimension = 128

results = {
    'sizes': vector_sizes,
    'brute_force': [],
    'faiss_flat': [],
    'faiss_ivf': []
}

for size in vector_sizes:
    print(f'\nTesting with {size} vectors and {n_queries} queries...')
    vectors = generate_sample_vectors(size, dimension)
    query_vectors = generate_sample_vectors(n_queries, dimension)

    start_time = time.time()
    for query in query_vectors:
        _ = brute_force_cosine_search(vectors, query, k)
    brute_force_time = (time.time() - start_time) / n_queries
    results['brute_force'].append(brute_force_time)

    start_time = time.time()
    for query in query_vectors:
        _ = faiss_flat_l2_search(vectors, query, k)
    faiss_flat_time = (time.time() - start_time) / n_queries
    results['faiss_flat'].append(faiss_flat_time)

    n_regions = min(size // 100, 1000)
    n_regions = max(n_regions, 1)
    _, search_time, _, _ = faiss_ivf_search_realistic(vectors, query_vectors, k, n_regions)
    faiss_ivf_time = search_time / n_queries
    results['faiss_ivf'].append(faiss_ivf_time)

    print('Average times per query (seconds):')
    print(f'Brute Force: {brute_force_time:.6f}')
    print(f'FAISS Flat: {faiss_flat_time:.6f}')
    print(f'FAISS IVF: {faiss_ivf_time:.6f}')

plt.figure(figsize=(12, 6))
plt.plot(results['sizes'], results['brute_force'], 'o-', label='Brute Force Cosine')
plt.plot(results['sizes'], results['faiss_flat'], 's-', label='FAISS Flat L2')
plt.plot(results['sizes'], results['faiss_ivf'], '^-', label='FAISS IVF')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Vectors')
plt.ylabel('Average Search Time per Query (seconds)')
plt.title('Search Time Comparison: Brute Force vs FAISS Methods')
plt.grid(True)
plt.legend()

for i, size in enumerate(results['sizes']):
    plt.annotate(f"{results['brute_force'][i]:.6f}", (size, results['brute_force'][i]), textcoords='offset points', xytext=(0, 10))
    plt.annotate(f"{results['faiss_flat'][i]:.6f}", (size, results['faiss_flat'][i]), textcoords='offset points', xytext=(0, -15))
    plt.annotate(f"{results['faiss_ivf'][i]:.6f}", (size, results['faiss_ivf'][i]), textcoords='offset points', xytext=(0, 10))

plt.tight_layout()
plt.show()
